# Feature engineering on landsat
This notebooks take the bands returned by landsat and calculates more features

## Environment preparation

In [ ]:
!pip install uv
!uv pip install  -r ../requirements.txt 

In [ ]:
# Data manipulation and analysis
import numpy as np
import pandas as pd

# Spatial feature
from sklearn.cluster import KMeans

# Importing functions
import sys
import os
sys.path.append(os.path.abspath('..'))
from utils import save_df, landsat_bands_of_interest, combine_datasets

In [ ]:
# Processing
eps = 1e-10

# Path
save_path = 'snow://workspace/USER$.PUBLIC."waterscript-ey-data-science-challenge"/versions/live/data/processed/'

# Cols not considered on removing high correlated ones
ignore_list = [
    # Target
    'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus',

    # Metadata
    'Latitude', 'Longitude', 'Sample Date', 'location_id', 'soil_name', 'kmeans_region'
]

In [ ]:
# Df for training
water_quality_df = (
    pd.read_csv('../data/raw/water_quality_training_dataset.csv')
)

landsat_train_features_base = (
    pd.read_csv('../data/intermediate/landsat_train_features_base.csv')
)

terraclimate_train_features_base = (
    pd.read_csv('../data/intermediate/terraclimate_train_features_base.csv')
)

# Df for testing
landsat_val_features_base = (
    pd.read_csv('../data/intermediate/landsat_val_features_base.csv')
)

terraclimate_val_features_base = (
    pd.read_csv('../data/intermediate/terraclimate_val_features_base.csv')
)

# Other static features
elevation_feature_df = (
    pd.read_csv('../data/intermediate/elevation_feature.csv')
)

soil_features_df = (
    pd.read_csv('../data/intermediate/soil_features.csv')
)

### Functions

In [ ]:
def create_date_features(df, date_column='Sample Date'):
    
    df_copy = df.copy()
    # Guarantees date
    df_copy[date_column] = pd.to_datetime(df_copy[date_column], dayfirst=True, errors='coerce')
    
    # Extract basic features
    df_copy['month'] = df_copy[date_column].dt.month
    df_copy['year'] = df_copy[date_column].dt.year
    df_copy['quarter'] = df_copy[date_column].dt.quarter
    
    # Ciclic transformation
    # Sen and Cos 'connects' December and January in a 'circle'
    df_copy['month_sin'] = np.sin(2 * np.pi * df_copy['month'] / 12)
    df_copy['month_cos'] = np.cos(2 * np.pi * df_copy['month'] / 12)
    
    return df_copy

In [ ]:
def apply_spatial_clusters(train_df, val_df, n_clusters=10):
    '''
    Create clusters based on lat and lon
    Guarantee that KMeans will be trained on both test and train dataset identically
    '''
    # KMeans algorithm
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    
    # Training
    kmeans.fit(train_df[['Latitude', 'Longitude']])
    
    # Predicting using same rule
    train_df['kmeans_region'] = kmeans.predict(train_df[['Latitude', 'Longitude']])
    val_df['kmeans_region'] = kmeans.predict(val_df[['Latitude', 'Longitude']])
    
    return train_df, val_df

In [ ]:
def create_climate_lags(df, date_column='Sample Date'):
    '''
    Create lags for ppt, q and soil for 1 and 3 months
    '''
    cols_to_lag = ['ppt', 'q', 'soil']
    df_copy = df.copy()
    
    # Sort by local and date
    df_copy = df_copy.sort_values(by=['location_id', date_column])
    
    # Applys shift grouped by local
    for col in cols_to_lag:
        # Takes the shift value for that river
        df_copy[f'{col}_lag1'] = df_copy.groupby('location_id')[col].shift(1)
        df_copy[f'{col}_lag3'] = df_copy.groupby('location_id')[col].shift(3)
    
    return df_copy.sort_index()

In [ ]:
def process_df(df):
    '''
    Calculates spectral indices and merges metadata from the original dataset.
    Indices documentation: https://docs.google.com/document/d/1VOdy3sTt4UJxjBM_zrndnOT1DKKQ_yHdgqNfwF0iKn4/edit?usp=sharing

    Calculates indices:
    Landsat:
    - NDMI (Normalized Difference Moisture Index): Measures vegetation water content. Formula: (NIR - SWIR16) / (NIR + SWIR16).
    - MNDWI (Modified Normalized Difference Water Index): Highlights open water features. Formula: (Green - SWIR16) / (Green + SWIR16).
    - NDTI (Normalized Difference Turbidity Index): Estimates water turbidity/cloudiness. Formula: (Red - Green) / (Red + Green).
    - NDSSI (Normalized Difference Suspended Sediment Index): Helps to detect thin sediments. Formula: (Blue - NIR) / (Blue + NIR).
    - Chlorophyll_Proxy: Estimates the F quantity. Formula: NIR / Green.
    - Red_Blue_Ratio: Helps to infer alcalinity. Formula: Red / Blue.
    - SI (Salinity Index): Measures the dissolved salts. Formula: np.sqrt(Blue * Red)
    - NDVI (Normalized Difference Vegetation Index): Detects near vegetation. Formula (NIR - Red) / (NIR + Red).

    TerraClimate:
    - Water_Balance (Index of Aridity): Relates precipitation to evapotranspiration to estimate dilution potential. Formula: (PPT + 1) / (PET + 1).
    - Water_Deficit (Net Water Deficit): Measures the absolute surplus or deficit of water in the month. Formula: PPT - PET.
    - Temp_Range (Diurnal Temperature Range): Difference between maximum and minimum temperature, a proxy for aridity. Formula: Tmax - Tmin.
    - Runoff_Ratio (Runoff Coefficient): Fraction of rainfall that becomes surface runoff, indicating soil saturation. Formula: (Q + 0.1) / (PPT + 1).
    - Soil_Wash (Soil Wash Potential): Interaction term capturing the potential for sediment transport during rain events on wet soil. Formula: Soil * PPT.
    - Rain_vs_Vegetation (Rainfall-Vegetation Interaction): Captures the "sponge effect" of vegetation. High rainfall on bare soil (low NDVI) increases turbidity
    and runoff, while high rainfall on dense vegetation (high NDVI) is filtered. Formula: PPT * (1 - NDVI).
    '''
    df_copy = df.copy()

    # Creates unique ID
    df_copy['location_id'] = (
        df_copy['Latitude'].astype(str) +
        '+' +
        df_copy['Longitude'].astype(str)
    )

    # Get date features - removed for testing
    # df_date_feats = create_date_features(df_copy)
    # df_feats = create_climate_lags(df_date_feats)

    df_feats = df_copy
    
    # Corrects the bands as the getting_started notebook shows (USGS)
    # Clip negative data to 0 for correct indices' calculus
    for band in landsat_bands_of_interest:
        if df_feats[band].mean() > 1:
            print(f"Corrigindo escala da banda {band}...")
            df_feats[band] = (df_feats[band] * 0.0000275) - 0.2
        df_feats[band] = df_feats[band].clip(0)

    # NDMI
    df_feats['NDMI'] = (
        (df_feats['nir08'] - df_feats['swir16'])
        / (df_feats['nir08'] + df_feats['swir16'] + eps)
    )
    
    # MNDWI
    df_feats['MNDWI'] = (
        (df_feats['green'] - df_feats['swir16'])
        / (df_feats['green'] + df_feats['swir16'] + eps)
    )
    
    # NDTI
    df_feats['NDTI'] = (
        (df_feats['red'] - df_feats['green'])
        / (df_feats['red'] + df_feats['green'] + eps)
    )
    
    # NDSSI
    df_feats['NDSSI'] = (
        (df_feats['blue'] - df_feats['nir08'])
        / (df_feats['blue'] + df_feats['nir08'] + eps)
    )

    # Chlorophyll_Proxy
    df_feats['Chlorophyll_Proxy'] = (
        df_feats['nir08'] / (df_feats['green'] + eps)
    )

    # Red_Blue_Ratio
    df_feats['Red_Blue_Ratio'] = (
        df_feats['red'] / (df_feats['blue'] + eps)
    )

    # Salinity Index
    df_feats['SI'] = np.sqrt(
        df_feats['blue'].clip(lower=0) * df_feats['red'].clip(lower=0)
    )

    # NDVI
    df_feats['NDVI'] = (
        (df_feats['nir08'] - df_feats['red'])
        / (df_feats['nir08'] + df_feats['red'] + eps)
    )

    # Water_Balance
    df_feats['Water_Balance'] = (
        (df_feats['ppt'] + 1)
        / (df_feats['pet'] + 1)
    )

    # Water_Deficit
    df_feats['Water_Deficit'] = (
        df_feats['ppt'] - df_feats['pet']
    )

    # Temp_Range
    df_feats['Temp_Range'] = (
        df_feats['tmax'] - df_feats['tmin']
    )

    # Runoff_Ratio
    df_feats['Runoff_Ratio'] = (
        (df_feats['q'] + 0.1)
        / (df_feats['ppt'] + 1)
    )

    # Soil_Wash
    df_feats['Soil_Wash'] = (
        df_feats['soil'] * df_feats['ppt']
    )

    # Clipping some features
    # DO NOT CHANGE: the model cannot forecast correctly without this
    # Some pixels may come with a slightly different than 0 numeric value
    features_to_clip = ['NDMI', 'MNDWI', 'NDTI', 'NDSSI', 'NDVI']
    for feature in features_to_clip:
        df_feats[feature] = df_feats[feature].clip(-1, 1)

    # Rain_vs_Vegetation
    df_feats['Rain_vs_Vegetation'] = (
        df_feats['ppt'] * (1 - df_feats['NDVI'])
    )

    return df_feats

In [ ]:
def remove_highly_correlated_features(df, threshold=0.95, ignore_cols=None):
    """
    Remove highly correlated features, keeping only one of them
    """
    if ignore_cols is None:
        ignore_cols = []

    # Selects only no ignored cols
    numeric_cols = [
        c for c in df.columns
        if c not in ignore_cols
        and pd.api.types.is_numeric_dtype(df[c])
    ]

    # Correlation dataframe
    df_corr = df[numeric_cols].corr().abs()

    # Selects upper triangle of correlation matrix (avoid analysing repeated or diagonal values)
    upper = (
        df_corr
        .where(
            np.triu(
                np.ones(df_corr.shape),
                k=1
            ).astype(bool)
        )
    )

    # Identifys columns with correlation gt trashold
    to_drop = [
        column for column in upper.columns
        if any(upper[column] > threshold)
    ]

    # Remove to_drop columns
    df_reduced = df.drop(columns=to_drop)
    
    return df_reduced, to_drop

## Data transformation

In [ ]:
# Combining base data and final data into a single dataset.
# Train
combined_train_df = combine_datasets(
    water_quality_df,
    [
        landsat_train_features_base,
        terraclimate_train_features_base
        # elevation_feature_df
        # soil_features_df
    ]
)

# Test
combined_val_df = combine_datasets(
    landsat_val_features_base,
    [
        terraclimate_val_features_base
        # elevation_feature_df,
        # soil_features_df
    ]
)

In [ ]:
train_features = process_df(combined_train_df)
val_features = process_df(combined_val_df)

# KMeans logic for both
# train_features_clustered, val_features_clustered = apply_spatial_clusters(train_features, val_features, n_clusters=15)

In [ ]:
train_df_reduced, dropped_features = remove_highly_correlated_features(
    train_features,
    threshold=0.95,
    ignore_cols=ignore_list
)

val_df_reduced = val_features.drop(columns=dropped_features)

## Data saving

In [ ]:
save_df(train_df_reduced, 'train_features', save_path)
save_df(val_df_reduced, 'val_features', save_path)